### Ранжирование
CatBoost получил 300 кандидатов и должен выдать финальный Топ-10, который увидит пользователь на экране.

**NDCG@10 (Normalized Discounted Cumulative Gain)**
* Учитывает две вещи: 1) Насколько релевантна игра, 2) На какой позиции она стоит.
Метрика nDCG (Normalized Discounted Cumulative Gain)
$$nDCG@K = \frac{DCG@K}{IDCG@K}$$
где:
$$DCG@K = \sum_{i=1}^{K} \frac{2^{rel_i} - 1}{\log_2(i + 1)}$$
или альтернативный (классический) вариант:
$$DCG@K = rel_1 + \sum_{i=2}^{K} \frac{rel_i}{\log_2(i)}$$
$$IDCG@K = \sum_{i=1}^{K} \frac{2^{rel_i^{ideal}} - 1}{\log_2(i + 1)}$$
**Что измеряет nDCG?**  
Качество ранжирования рекомендаций с учётом **двух факторов**:

1. **Релевантность** — насколько рекомендованный элемент полезен пользователю
2. **Позиция** — чем выше в списке релевантный элемент, тем лучше (пользователь скорее его увидит)


**Вспомогательная метрика: MAP@10 (Mean Average Precision)**


$$MAP@K = \frac{1}{|U|} \sum_{u=1}^{|U|} AP@K(u)$$

где для каждого пользователя $u$:

$$AP@K(u) = \frac{1}{\min(K, |\text{RelevantItems}_u|)} \sum_{i=1}^{K} P@i(u) \cdot rel_i(u)$$

**Что измеряет MAP@K?**  
Среднее значение точности на тех позициях, где встречаются релевантные элементы, усреднённое по всем пользователям.
MAP штрафует модель за то, что релевантные элементы располагаются **низко** в списке рекомендаций.


#### 1. Novelty (Новизна / Непопулярность)
* **Что это:** Насколько "нишевые" игры мы рекомендуем.
* **Как считать:** Для каждой игры $i$ вычисляем вероятность ее встречи $p(i)$ (доля юзеров, игравших в нее, от всех юзеров). Новизна рекомендации — это $ - \log_2(p(i)) $. Чем популярнее игра (например, CS:GO), тем ближе метрика к 0. Чем реже игра — тем выше скор. Среднее по Топ-10 даст метрику `Mean_Novelty`.
* **Зачем:** Чтобы система помогала пользователям делать **открытия (Discovery)**, а не кормила их мейнстримом.

#### 2. Intra-List Diversity (Разнообразие внутри списка)
* **Что это:** Насколько разные игры лежат в Топ-10 *у конкретного пользователя*.
* **Как считать:** Берем признаки игр (например, жанры или теги). Для каждой пары игр в Топ-10 считаем косинусное расстояние между ними. Усредняем.
  * Если в Топ-10 лежат 10 "Шутеров от первого лица" -> Diversity близко к 0 (плохо).
  * Если там есть Шутер, RPG, Стратегия и Инди-платформер -> Diversity высокая (хорошо).
* **Зачем:** Если юзеру не хочется сейчас стрелять, а мы выдали 10 шутеров — мы потеряли юзера. Разнообразие страхует нас от ошибки узкого профилирования.

Выделение Таргета и Time-based Сплит данных
1) Отсекаем "холодных" пользователей (менее 5 игр).
2) Сортируем историю взаимодействий каждого пользователя по времени (rtime_last_played).
3) Делим данные пользователя на History (первые 80% по времени — пойдут на генерацию признаков) и Future (последние 20% — то, что он реально сыграл).
4) Размечаем кандидатов от ALS: если игра из als_candidates попала в список Future, присваиваем ей target из таблицы взаимодействий. Иначе target = 0 (негативный пример).
5) Делим пользователей на train_users и test_users для честной валидации CatBoost. Сохраняем всё в папку artifacts/ для Streamlit.


In [4]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

os.makedirs('artifacts', exist_ok=True)

interactions = pd.read_csv('data/user_games_target.csv')
candidates = pd.read_csv('data/als_candidates_top300.csv')

user_counts = interactions['steamid'].value_counts()
warm_users = user_counts[user_counts >= 5].index

interactions = interactions[interactions['steamid'].isin(warm_users)].copy()
candidates = candidates[candidates['steamid'].isin(warm_users)].copy()

interactions = interactions.sort_values(['steamid', 'rtime_last_played'])
interactions['rank_time'] = interactions.groupby('steamid').cumcount() + 1
interactions['total_games'] = interactions.groupby('steamid')['steamid'].transform('count')
interactions['is_future'] = interactions['rank_time'] > (interactions['total_games'] * 0.8)

history_interactions = interactions[~interactions['is_future']].drop(columns=['rank_time', 'total_games', 'is_future'])
future_interactions = interactions[interactions['is_future']].drop(columns=['rank_time', 'total_games', 'is_future'])

history_interactions.to_parquet('artifacts/history_interactions.parquet', index=False)

future_targets = future_interactions[['steamid', 'appid', 'target']]
reranker_data = candidates.merge(future_targets, on=['steamid', 'appid'], how='left')
reranker_data['target'] = reranker_data['target'].fillna(0)

unique_warm_users = reranker_data['steamid'].unique()
train_users, test_users = train_test_split(unique_warm_users, test_size=0.2, random_state=42)

df_train = reranker_data[reranker_data['steamid'].isin(train_users)]
df_test = reranker_data[reranker_data['steamid'].isin(test_users)]

df_train.to_parquet('artifacts/train_reranker_base.parquet', index=False)
df_test.to_parquet('artifacts/test_reranker_base.parquet', index=False)

1) Берем сырые данные о пользователях (unique_users.csv) и нашу историческую часть взаимодействий (history_interactions.parquet).
2) Считаем статистику по истории: сколько всего игр, суммарное и среднее время в игре.
3) Вычисляем возраст аккаунта в днях на момент последнего исторического взаимодействия (max(rtime) - timecreated).
4) Оставляем только нужные фичи (страну и рассчитанную статистику), заполняем пропуски.

In [5]:
users = pd.read_csv('data/unique_users.csv')
history = pd.read_parquet('artifacts/history_interactions.parquet')

user_static = users[['steamid', 'loccountrycode', 'timecreated']].copy()
user_static['loccountrycode'] = user_static['loccountrycode'].fillna('UNKNOWN')

user_behavior = history.groupby('steamid').agg(
    user_total_games_played=('appid', 'count'),
    user_total_playtime=('playtime_forever', 'sum'),
    user_avg_playtime=('playtime_forever', 'mean'),
    max_history_rtime=('rtime_last_played', 'max')
).reset_index()

user_features = user_static.merge(user_behavior, on='steamid', how='right')

user_features['account_age_days'] = (user_features['max_history_rtime'] - user_features['timecreated']) / 86400
user_features['account_age_days'] = user_features['account_age_days'].fillna(0).clip(lower=0)

user_features = user_features.drop(columns=['timecreated', 'max_history_rtime'])

user_features.to_parquet('artifacts/user_features.parquet', index=False)

Что делаем:
1) Преобразуем is_free в бинарный флаг (1/0).
2) Считаем recommendations_log через np.log1p, чтобы сгладить выбросы (Long Tail).
3) Вычисляем age_years (возраст игры в годах) из release_date. Для невалидных дат заполняем нулями.
4) Считаем количество поддерживаемых платформ (platforms_count), суммируя булевы флаги.
5) Очищаем массивы genres и categories от спецсимволов ([, ], "). Оставляем их как строку со словами, разделенными пробелом — CatBoost съест это как тип признака Text и сам извлечет эмбеддинги.
6) Сохраняем в item_features.parquet для дальнейшего объединения.

In [6]:
"""games = pd.read_csv('data/game_details.csv')

item_features = games[['appid', 'is_free', 'recommendations_total']].copy()
item_features['is_free'] = item_features['is_free'].fillna(False).astype(int)
item_features['recommendations_log'] = np.log1p(item_features['recommendations_total'].fillna(0))
item_features = item_features.drop(columns=['recommendations_total'])

release_dates = pd.to_datetime(games['release_date'], errors='coerce')
item_features['age_years'] = (pd.Timestamp.now() - release_dates).dt.days / 365.25
item_features['age_years'] = item_features['age_years'].fillna(0).clip(lower=0)

item_features['platforms_count'] = (
    games['platforms_windows'].fillna(False).astype(int) +
    games['platforms_mac'].fillna(False).astype(int) +
    games['platforms_linux'].fillna(False).astype(int)
)

for col in ['genres', 'categories']:
    item_features[col] = games[col].fillna('') \
        .str.replace(r'\[|\]|"', '', regex=True) \
        .str.replace(',', ' ') \
        .str.strip()

item_features.to_parquet('artifacts/item_features.parquet', index=False)"""

'games = pd.read_csv(\'data/game_details.csv\')\n\nitem_features = games[[\'appid\', \'is_free\', \'recommendations_total\']].copy()\nitem_features[\'is_free\'] = item_features[\'is_free\'].fillna(False).astype(int)\nitem_features[\'recommendations_log\'] = np.log1p(item_features[\'recommendations_total\'].fillna(0))\nitem_features = item_features.drop(columns=[\'recommendations_total\'])\n\nrelease_dates = pd.to_datetime(games[\'release_date\'], errors=\'coerce\')\nitem_features[\'age_years\'] = (pd.Timestamp.now() - release_dates).dt.days / 365.25\nitem_features[\'age_years\'] = item_features[\'age_years\'].fillna(0).clip(lower=0)\n\nitem_features[\'platforms_count\'] = (\n    games[\'platforms_windows\'].fillna(False).astype(int) +\n    games[\'platforms_mac\'].fillna(False).astype(int) +\n    games[\'platforms_linux\'].fillna(False).astype(int)\n)\n\nfor col in [\'genres\', \'categories\']:\n    item_features[col] = games[col].fillna(\'\')         .str.replace(r\'\\[|\\]|"\', \'\',

вместо 5, 6 - Multi-Hot Encoding для Топ-20 популярных жанров и категорий:

In [7]:
import ast
from collections import Counter

games = pd.read_csv('data/game_details.csv')
item_features_mh = games[['appid', 'is_free', 'recommendations_total']].copy()

item_features_mh['is_free'] = item_features_mh['is_free'].fillna(False).astype(int)
item_features_mh['recommendations_log'] = np.log1p(item_features_mh['recommendations_total'].fillna(0))
item_features_mh = item_features_mh.drop(columns=['recommendations_total'])

release_dates = pd.to_datetime(games['release_date'], errors='coerce')
item_features_mh['age_years'] = (pd.Timestamp.now() - release_dates).dt.days / 365.25
item_features_mh['age_years'] = item_features_mh['age_years'].fillna(0).clip(lower=0)

item_features_mh['platforms_count'] = (
    games['platforms_windows'].fillna(False).astype(int) +
    games['platforms_mac'].fillna(False).astype(int) +
    games['platforms_linux'].fillna(False).astype(int)
)

def extract_top_k_multihot(df, source_df, col, k=20):
    cleaned = source_df[col].fillna('[]').str.replace('""', '"')
    parsed = cleaned.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else [])
    
    all_elements = [item for sublist in parsed for item in sublist]
    top_k = [x[0] for x in Counter(all_elements).most_common(k)]
    
    for item in top_k:
        safe_name = item.replace(' ', '_').replace('/', '_')
        df[f'{col}_{safe_name}'] = parsed.apply(lambda x: 1 if item in x else 0).astype('int8')
        
    return df, top_k

item_features_mh, top_genres = extract_top_k_multihot(item_features_mh, games, 'genres', 20)
item_features_mh, top_categories = extract_top_k_multihot(item_features_mh, games, 'categories', 20)

pd.Series(top_genres).to_csv('artifacts/top_20_genres.csv', index=False)
pd.Series(top_categories).to_csv('artifacts/top_20_categories.csv', index=False)
item_features_mh.to_parquet('artifacts/item_features_multihot.parquet', index=False)

### User-Item Interaction Features, Candidate Features и Сборка финального датасета

1. **User-Item Affinity:** Вычисляем историческую склонность юзера к жанрам/категориям. Мы джоиним историю юзера с Multi-Hot признаками игр, суммируем их и делим на общее число сыгранных игр. Получаем долю игр в каждом жанре для каждого юзера (например, если юзер сыграл 10 игр и 8 из них Action, то `user_affinity_genres_Action = 0.8`).
2. **Candidate Features:** `score` и `rank` от ALS уже находятся в базовых файлах трейна/теста (мы их сджоинили на первом этапе). Мы просто переименуем их для понятности.
3. Собираем всё вместе: База (ALS кандидаты) + User Features + Item Features + User Affinity. Формируем финальные матрицы для обучения.


In [8]:
history = pd.read_parquet('artifacts/history_interactions.parquet')
user_feat = pd.read_parquet('artifacts/user_features.parquet')
item_feat_mh = pd.read_parquet('artifacts/item_features_multihot.parquet')

history_with_items = history[['steamid', 'appid']].merge(item_feat_mh, on='appid', how='inner')

mh_cols = [c for c in item_feat_mh.columns if c.startswith('genres_') or c.startswith('categories_')]
user_affinity = history_with_items.groupby('steamid')[mh_cols].sum().reset_index()

user_game_counts = history.groupby('steamid').size().reset_index(name='games_count')
user_affinity = user_affinity.merge(user_game_counts, on='steamid')

for col in mh_cols:
    user_affinity[f'user_affinity_{col}'] = (user_affinity[col] / user_affinity['games_count']).astype('float32')

user_affinity = user_affinity[['steamid'] + [f'user_affinity_{c}' for c in mh_cols]]
user_affinity.to_parquet('artifacts/user_affinity.parquet', index=False)

def assemble_final_dataset(base_file_path):
    df_base = pd.read_parquet(base_file_path)
    df_base = df_base.rename(columns={'score': 'als_score', 'rank': 'als_rank'})
    
    df_final = df_base.merge(user_feat, on='steamid', how='left')
    df_final = df_final.merge(user_affinity, on='steamid', how='left')
    df_final = df_final.merge(item_feat_mh, on='appid', how='left')
    
    affinity_cols = [c for c in df_final.columns if c.startswith('user_affinity_')]
    df_final[affinity_cols] = df_final[affinity_cols].fillna(0)
    
    return df_final

train_final = assemble_final_dataset('artifacts/train_reranker_base.parquet')
test_final = assemble_final_dataset('artifacts/test_reranker_base.parquet')

train_final = train_final.sort_values('steamid')
test_final = test_final.sort_values('steamid')

train_final.to_parquet('artifacts/train_final.parquet', index=False)
test_final.to_parquet('artifacts/test_final.parquet', index=False)

### Отчет по сгенерированным признакам (Feature Engineering Report)

В ходе обработки сырых данных и применения Time-based сплита был сформирован итоговый датасет (`train_final.parquet` / `test_final.parquet`). Ниже представлен полный список признаков, которые будут поданы в модель CatBoost, с описанием их обработки:

#### 1. Таргет и признаки кандидата (От модели 1-го уровня)
*   **`target`** (Целевая переменная): Заполняется из тестовой (Future) части логов пользователя. Если игра из выдачи ALS была сыграна пользователем в будущем — таргет > 0 (от 1 до 5 по перцентилям playtime). Если не сыграна — `0` (хард-негатив).
*   **`als_score`** (Числовой): Оценка (уверенность) алгоритма ALS. Переименован из колонки `score` таблицы кандидатов.
*   **`als_rank`** (Числовой): Позиция (1-300) кандидата в выдаче ALS. Переименован из колонки `rank`.

#### 2. Признаки пользователя (User Features)
*Формируются строго по исторической части данных (History), чтобы избежать заглядывания в будущее.*
*   **`loccountrycode`** (Категориальный): Код страны пользователя. Пропуски заполнены константой `UNKNOWN`.
*   **`user_total_games_played`** (Числовой): Общее количество уникальных игр в истории пользователя.
*   **`user_total_playtime`** (Числовой): Суммарное время, проведенное пользователем во всех играх.
*   **`user_avg_playtime`** (Числовой): Среднее время сессии пользователя по его истории.
*   **`account_age_days`** (Числовой): Жизненный цикл аккаунта в днях (разница между последней активностью в истории и `timecreated`). Отрицательные значения и пропуски отсечены до `0`. Мусорные признаки (`profileurl`, `avatar`, `realname`) удалены.

#### 3. Признаки игры (Item Features)
*   **`is_free`** (Бинарный): Флаг бесплатной игры (`1` - бесплатная, `0` - платная). Конвертирован из булевых значений.
*   **`recommendations_log`** (Числовой): Логарифмированное количество рекомендаций (`np.log1p(recommendations_total)`). Применено для сглаживания выбросов (борьба с Long Tail).
*   **`age_years`** (Числовой): Возраст игры в годах на текущий момент. Вычислен из `release_date`. Некорректные даты переведены в `0`.
*   **`platforms_count`** (Числовой от 0 до 3): Общее количество поддерживаемых платформ (сумма флагов Windows, Mac, Linux).
*   **`genres_*` (20 признаков)** (Бинарные): Multi-Hot кодировка Топ-20 самых популярных жанров в Steam (например, `genres_Action`, `genres_Indie`). Строковые массивы были распаршены через `ast.literal_eval`.
*   **`categories_*` (20 признаков)** (Бинарные): Multi-Hot кодировка Топ-20 самых популярных категорий (например, `categories_Multi-player`, `categories_Co-op`).

#### 4. Исторические пересечения (User-Item Affinity)
*Ключевые признаки (Килеры) для бустинга, показывающие склонность пользователя к характеристикам конкретной игры.*
*   **`user_affinity_genres_*` (20 признаков)** (Числовой от 0.0 до 1.0): Доля игр определенного жанра в истории пользователя. (Например, если из 10 сыгранных игр 8 были Action, признак `user_affinity_genres_Action` = 0.8).
*   **`user_affinity_categories_*` (20 признаков)** (Числовой от 0.0 до 1.0): Доля игр определенной категории в истории пользователя (вычисляется аналогично жанрам). Пропуски (если пользователь вообще не играл в игры с известными тегами) заполнены нулями.

### 4. Архитектура Fallback (Эвристика для "холодных" пользователей)

**Что делаем:**
1. Для пользователей с историей < 5 игр ML-модели будут выдавать шум. Для них мы рассчитываем статический топ самых популярных игр.
2. Чтобы избежать "утечки данных", популярность (количество игроков) считаем **только по исторической части взаимодействий** (`history_interactions`).
3. Разделяем топ на Бесплатные (Топ-20) и Платные (Топ-10), так как `is_free` — сильный фактор для пользователя.
4. Сохраняем эти списки в виде словаря в файл `fallback_recommendations.pkl` внутри `artifacts/`. Приложение на Streamlit будет загружать этот файл и мгновенно отдавать дефолтные рекомендации "холодным" юзерам без запуска тяжелых моделей.


In [9]:
import pickle

history = pd.read_parquet('artifacts/history_interactions.parquet')
item_features = pd.read_parquet('artifacts/item_features_multihot.parquet')

game_popularity = history.groupby('appid').size().reset_index(name='players_count')
game_pop_features = game_popularity.merge(item_features[['appid', 'is_free']], on='appid', how='inner')

top_free = game_pop_features[game_pop_features['is_free'] == 1] \
    .sort_values('players_count', ascending=False).head(20)['appid'].tolist()

top_paid = game_pop_features[game_pop_features['is_free'] == 0] \
    .sort_values('players_count', ascending=False).head(10)['appid'].tolist()

general_fallback = top_free[:15] + top_paid[:15]

fallback_dict = {
    'top_free': top_free,
    'top_paid': top_paid,
    'general_fallback': general_fallback
}

with open('artifacts/fallback_recommendations.pkl', 'wb') as f:
    pickle.dump(fallback_dict, f)

def get_fallback_recommendations(top_k=10):
    return general_fallback[:top_k]

In [10]:
%pip install mlflow

  Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl (427 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 6.33.6 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import mlflow
import mlflow.catboost
from catboost import CatBoostRanker, Pool
from catboost.utils import get_gpu_device_count

# Проверка, видит ли CatBoost GPU
print("GPU devices visible to CatBoost:", get_gpu_device_count())

train = pd.read_parquet("artifacts/train_final.parquet").sort_values("steamid")
test = pd.read_parquet("artifacts/test_final.parquet").sort_values("steamid")

drop_cols = ["steamid", "appid", "target"]
X_train = train.drop(columns=drop_cols)
y_train = train["target"]
q_train = train["steamid"]

X_test = test.drop(columns=drop_cols)
y_test = test["target"]
q_test = test["steamid"]

cat_features = ["loccountrycode"]

train_pool = Pool(
    data=X_train,
    label=y_train,
    group_id=q_train,
    cat_features=cat_features
)

test_pool = Pool(
    data=X_test,
    label=y_test,
    group_id=q_test,
    cat_features=cat_features
)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Steam_RecSys_Reranking")

with mlflow.start_run(run_name="CatBoost_PairLogitPairwise_GPU"):
    params = {
        "iterations": 500,
        "learning_rate": 0.1,
        "depth": 5,  # для PairLogitPairwise на GPU максимум 8
        "loss_function": "PairLogitPairwise",
        "custom_metric": ["NDCG:top=10", "MAP:top=10"],
        "eval_metric": "NDCG:top=10",
        "early_stopping_rounds": 50,
        "random_seed": 42,
        "task_type": "GPU",
        "devices": "0",
        "border_count": 32,
        "bootstrap_type": "Bayesian",
        "verbose": 200,
    }

    mlflow.log_params(params)

    model = CatBoostRanker(**params)
    model.fit(train_pool, eval_set=test_pool, verbose=200)

    best_score = model.get_best_score()
    best_iteration = model.get_best_iteration()
    best_ndcg_10 = best_score["validation"]["NDCG:top=10;type=Base"]

    mlflow.log_metric("best_iteration", best_iteration)
    mlflow.log_metric("best_ndcg_10", best_ndcg_10)

    model.save_model("artifacts/catboost_ranker.cbm")
    mlflow.catboost.log_model(model, artifact_path="model")

    print(f"Best iteration: {best_iteration}")
    print(f"Best NDCG@10: {best_ndcg_10:.4f}")

GPU devices visible to CatBoost: 0


CatBoostError: /Users/zomb-ml-platform-msk/go-agent-21.2.0/pipelines/BuildMaster/catboost.git/catboost/libs/train_lib/trainer_env.cpp:9: Environment for task type [GPU] not found

In [ ]:
"""import optuna
import mlflow
import mlflow.catboost
from catboost import CatBoostRanker
import time

def objective(trial):
    start_time = time.time()
    print(f"\n{'='*50}")
    print(f"Trial {trial.number + 1} начат")
    
    params = {
        'iterations': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'depth': trial.suggest_int('depth', 4, 9),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 3.0, 5.0),
        'loss_function': 'PairLogitPairwise',
        'eval_metric': 'NDCG:top=10',
        'early_stopping_rounds': 15,
        'random_seed': 42,
        'verbose': False,
        'task_type': 'GPU',
        'thread_count': 4,
        'bootstrap_type': 'Bayesian'
    }
    
    print(f"lr={params['learning_rate']:.4f}, depth={params['depth']}, l2={params['l2_leaf_reg']:.2f}")
    
    model = CatBoostRanker(**params)
    model.fit(train_pool, eval_set=test_pool)
    
    score = model.get_best_score()['validation']['NDCG:top=10;type=Base']
    elapsed = time.time() - start_time
    
    # Исправление: проверяем есть ли завершенные trials
    if len(study.trials) > 1:
        print(f"Trial {trial.number + 1} завершен за {elapsed:.0f}с. NDCG@10 = {score:.4f} (лучший: {study.best_value:.4f})")
    else:
        print(f"Trial {trial.number + 1} завершен за {elapsed:.0f}с. NDCG@10 = {score:.4f}")
    
    return score

print("="*50)
print("ЗАПУСК OPTUNA ДЛЯ ПОДБОРА ГИПЕРПАРАМЕТРОВ")
print(f"Всего trials: 20")
print("="*50)

study = optuna.create_study(direction="maximize", study_name="CatBoost_Reranker_Tuning")
study.optimize(objective, n_trials=20)

print("\n" + "="*50)
print("ПОДБОР ГИПЕРПАРАМЕТРОВ ЗАВЕРШЕН")
print(f"Лучший NDCG@10: {study.best_value:.4f}")
print(f"Лучшие параметры: {study.best_params}")
print("="*50)

print("\n" + "="*50)
print("ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ С ЛУЧШИМИ ПАРАМЕТРАМИ")
print("="*50)

with mlflow.start_run(run_name="CatBoost_Optuna_Best"):
    best_params = study.best_params
    best_params['iterations'] = 300
    best_params['loss_function'] = 'PairLogitPairwise'
    best_params['eval_metric'] = 'NDCG:top=10;type=Base'
    best_params['early_stopping_rounds'] = 30
    best_params['random_seed'] = 42
    best_params['task_type'] = 'GPU'
    best_params['thread_count'] = 4
    best_params['bootstrap_type'] = 'Bayesian'
    
    print(f"Финальные параметры:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    
    mlflow.log_params(best_params)
    
    print("\nОбучение модели...")
    best_model = CatBoostRanker(**best_params)
    best_model.fit(train_pool, eval_set=test_pool, verbose=50)
    
    best_ndcg = best_model.get_best_score()['validation']['NDCG:top=10;type=Base']
    print(f"\nФинальный NDCG@10: {best_ndcg:.4f}")
    print(f"Лучшая итерация: {best_model.get_best_iteration()}")
    
    mlflow.log_metric("best_ndcg_10", best_ndcg)
    mlflow.log_metric("optuna_trials", len(study.trials))
    
    print("\nСохранение модели...")
    best_model.save_model('artifacts/catboost_ranker_tuned.cbm')
    mlflow.catboost.log_model(best_model, artifact_path="model_tuned")
    
    print("Модель сохранена!")
    print("="*50)"""

'import optuna\nimport mlflow\nimport mlflow.catboost\nfrom catboost import CatBoostRanker\nimport time\n\ndef objective(trial):\n    start_time = time.time()\n    print(f"\n{\'=\'*50}")\n    print(f"Trial {trial.number + 1} начат")\n\n    params = {\n        \'iterations\': 1000,\n        \'learning_rate\': trial.suggest_float(\'learning_rate\', 0.01, 0.2),\n        \'depth\': trial.suggest_int(\'depth\', 4, 9),\n        \'l2_leaf_reg\': trial.suggest_float(\'l2_leaf_reg\', 3.0, 5.0),\n        \'loss_function\': \'PairLogitPairwise\',\n        \'eval_metric\': \'NDCG:top=10\',\n        \'early_stopping_rounds\': 15,\n        \'random_seed\': 42,\n        \'verbose\': False,\n        \'task_type\': \'GPU\',\n        \'thread_count\': 4,\n        \'bootstrap_type\': \'Bayesian\'\n    }\n\n    print(f"lr={params[\'learning_rate\']:.4f}, depth={params[\'depth\']}, l2={params[\'l2_leaf_reg\']:.2f}")\n\n    model = CatBoostRanker(**params)\n    model.fit(train_pool, eval_set=test_pool)\n\n 

In [ ]:
import os
import optuna
import mlflow
import mlflow.catboost
from catboost import CatBoostRanker
from catboost.utils import get_gpu_device_count
import time

print(f"GPU devices visible to CatBoost: {get_gpu_device_count()}")

def objective(trial):
    start_time = time.time()
    print(f"\n{'='*50}")
    print(f"Trial {trial.number + 1} начат")

    params = {
        "iterations": 1200,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.5),
        "depth": trial.suggest_int("depth", 4, 8),  # на GPU для pairwise-модов максимум 8
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 3.0, 5.0),
        "loss_function": "PairLogitPairwise",
        "eval_metric": "NDCG:top=10",
        "early_stopping_rounds": 100,
        "random_seed": 42,
        "verbose": False,
        "task_type": "GPU",
        "devices": "0",
        "border_count": 32,
        "bootstrap_type": "Bayesian",
    }

    print(f"lr={params['learning_rate']:.4f}, depth={params['depth']}, l2={params['l2_leaf_reg']:.2f}")

    model = CatBoostRanker(**params)
    model.fit(train_pool, eval_set=test_pool)

    score = model.get_best_score()["validation"]["NDCG:top=10;type=Base"]
    elapsed = time.time() - start_time

    if len(study.trials) > 1:
        print(f"Trial {trial.number + 1} завершен за {elapsed:.0f}с. NDCG@10 = {score:.4f} (лучший: {study.best_value:.4f})")
    else:
        print(f"Trial {trial.number + 1} завершен за {elapsed:.0f}с. NDCG@10 = {score:.4f}")

    return score


print("="*50)
print("ЗАПУСК OPTUNA ДЛЯ ПОДБОРА ГИПЕРПАРАМЕТРОВ")
print("Всего trials: 20")
print("="*50)

study = optuna.create_study(direction="maximize", study_name="CatBoost_Ranker_Tuning")
study.optimize(objective, n_trials=50)

print("\n" + "="*50)
print("ПОДБОР ГИПЕРПАРАМЕТРОВ ЗАВЕРШЕН")
print(f"Лучший NDCG@10: {study.best_value:.4f}")
print(f"Лучшие параметры: {study.best_params}")
print("="*50)

print("\n" + "="*50)
print("ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ С ЛУЧШИМИ ПАРАМЕТРАМИ")
print("="*50)

with mlflow.start_run(run_name="CatBoost_Optuna_Best"):
    best_params = study.best_params.copy()
    best_params["iterations"] = 300
    best_params["loss_function"] = "PairLogitPairwise"
    best_params["eval_metric"] = "NDCG:top=10;type=Base"
    best_params["early_stopping_rounds"] = 30
    best_params["random_seed"] = 42
    best_params["task_type"] = "GPU"
    best_params["devices"] = "0"
    best_params["border_count"] = 32
    best_params["bootstrap_type"] = "Bayesian"
    best_params["verbose"] = 50

    print("Финальные параметры:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")

    mlflow.log_params(best_params)

    print("\nОбучение модели...")
    best_model = CatBoostRanker(**best_params)
    best_model.fit(train_pool, eval_set=test_pool, verbose=50)

    best_ndcg = best_model.get_best_score()["validation"]["NDCG:top=10;type=Base"]
    print(f"\nФинальный NDCG@10: {best_ndcg:.4f}")
    print(f"Лучшая итерация: {best_model.get_best_iteration()}")

    mlflow.log_metric("best_ndcg_10", best_ndcg)
    mlflow.log_metric("optuna_trials", len(study.trials))

    print("\nСохранение модели...")
    best_model.save_model("artifacts/catboost_ranker_tuned.cbm")
    mlflow.catboost.log_model(best_model, artifact_path="model_tuned")

    print("Модель сохранена!")
    print("="*50) 

[I 2026-05-14 09:09:40,891] A new study created in memory with name: CatBoost_Ranker_Tuning


GPU devices visible to CatBoost: 1
ЗАПУСК OPTUNA ДЛЯ ПОДБОРА ГИПЕРПАРАМЕТРОВ
Всего trials: 20

Trial 1 начат
lr=0.4285, depth=4, l2=4.92


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
[I 2026-05-14 09:09:44,505] Trial 0 finished with value: 0.36368790913072374 and parameters: {'learning_rate': 0.4285034616881178, 'depth': 4, 'l2_leaf_reg': 4.924736653609715}. Best is trial 0 with value: 0.36368790913072374.


Trial 1 завершен за 4с. NDCG@10 = 0.3637

Trial 2 начат
lr=0.3817, depth=6, l2=4.38


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
[I 2026-05-14 09:09:49,696] Trial 1 finished with value: 0.36492949978801775 and parameters: {'learning_rate': 0.3817156016006872, 'depth': 6, 'l2_leaf_reg': 4.380752367827818}. Best is trial 1 with value: 0.36492949978801775.


...

Trial 50 завершен за 132с. NDCG@10 = 0.3806 (лучший: 0.3831)

ПОДБОР ГИПЕРПАРАМЕТРОВ ЗАВЕРШЕН
Лучший NDCG@10: 0.3831
Лучшие параметры: {'learning_rate': 0.10446579006713684, 'depth': 8, 'l2_leaf_reg': 3.6783434411767573}

ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ С ЛУЧШИМИ ПАРАМЕТРАМИ
Финальные параметры:
  learning_rate: 0.10446579006713684
  depth: 8
  l2_leaf_reg: 3.6783434411767573
  iterations: 300
  loss_function: PairLogitPairwise
  eval_metric: NDCG:top=10;type=Base
  early_stopping_rounds: 30
  random_seed: 42
  task_type: GPU
  devices: 0
  border_count: 32
  bootstrap_type: Bayesian
  verbose: 50

Обучение модели...
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2449403	best: 0.2449403 (0)	total: 288ms	remaining: 1m 26s
50:	test: 0.3495316	best: 0.3495316 (50)	total: 14.5s	remaining: 1m 10s
100:	test: 0.3633661	best: 0.3633661 (100)	total: 28.7s	remaining: 56.5s
150:	test: 0.3720222	best: 0.3720222 (150)	total: 42.8s	remaining: 42.3s


2026/05/14 10:15:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


bestTest = 0.3720222019
bestIteration = 150
Shrink model to first 151 iterations.

Финальный NDCG@10: 0.3720
Лучшая итерация: 150

Сохранение модели...
Модель сохранена!


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

# Предполагается, что best_model и test_pool у тебя уже есть из предыдущего кода

def analyze_model_adequacy(model, pool, top_n=20):
    print("="*50)
    print("1. БАЗОВАЯ ВАЖНОСТЬ ПРИЗНАКОВ (Feature Importance)")
    print("="*50)
    
    # Получаем важность и имена признаков
    importances = model.get_feature_importance(pool)
    feature_names = pool.get_feature_names()
    
    # Собираем в DataFrame и сортируем
    fi_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    
    # Рисуем график
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=fi_df.head(top_n), palette='viridis')
    plt.title(f'Топ-{top_n} признаков (LossFunctionChange)')
    plt.tight_layout()
    plt.show()
    
    # Логируем в MLflow (если активен run)
    if mlflow.active_run():
        # Сохраняем топ фичей в csv и кидаем в артефакты
        fi_df.to_csv("feature_importance.csv", index=False)
        mlflow.log_artifact("feature_importance.csv")
    
    print("="*50)
    print("2. SHAP VALUES (Направление влияния признаков)")
    print("="*50)
    
    # Извлекаем признаки из pool
    X_features = pd.DataFrame(pool.get_features(), columns=feature_names)
    
    # Для SHAP расчетов берем случайную подвыборку, чтобы не ждать вечность, 
    # если test_pool огромный (например, 10 000 строк)
    X_sample = X_features.sample(min(10000, len(X_features)), random_state=42)
    
    # Считаем SHAP значения
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # Строим Summary Plot
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_sample, plot_type="dot", show=False)
    plt.title("SHAP Summary Plot")
    plt.tight_layout()
    plt.show()

# Запускаем функцию (используем test_pool для честной оценки)
analyze_model_adequacy(best_model, test_pool)